# 💊 Plain-Language Drug Labels

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pranavladha612/med-label-simplifier/blob/main/colab_demo.ipynb)

Rewrites medication labels at a **~5th-grade reading level** with an LLM, then **verifies that no dose, time limit, age limit or warning was lost**, and that nothing was invented.

Rewriting is easy for an LLM. The hard part, and the point of this project, is **proving the rewrite is still safe**. Two independent checks do that:

1. **Fact check:** extract every dose, time, age and warning from the original and confirm each one survives the rewrite.
2. **Meaning check (NLI):** a local AI model checks, sentence by sentence, whether the rewrite says anything the original doesn't, or contradicts it.

If either check finds a problem, the LLM gets a "fix-it" prompt listing exactly what went wrong and tries again.

> ⚠️ **Not medical advice.** Research prototype. Simplified text must be reviewed by a pharmacist.

**How to use this notebook:** run the cells top to bottom (`Runtime → Run all`).
Parts 1 and 2 work **without any API key**. Part 3 onward needs a free [OpenRouter](https://openrouter.ai/keys) key (see Setup).

Code: [github.com/Pranavladha612/med-label-simplifier](https://github.com/Pranavladha612/med-label-simplifier)

## Setup

In [1]:
# @title Install and load the project (about 1 minute)
import os, sys, warnings
warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_VERBOSITY"] = "error"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("med-label-simplifier"):
        !git clone -q https://github.com/Pranavladha612/med-label-simplifier.git
    %cd med-label-simplifier
    !pip install -q openai textstat python-dotenv

from IPython.display import HTML, display
print("Ready." + (" (running in Colab)" if IN_COLAB else " (running locally)"))

Ready. (running locally)


### API key (only needed from Part 3)

1. Get a free key at [openrouter.ai/keys](https://openrouter.ai/keys).
2. In Colab, click the **🔑 key icon** in the left sidebar → **Add new secret**.
3. Name it `OPENROUTER_API_KEY`, paste the key as the value, and switch on **Notebook access**.

The key stays in your Colab account. It never appears in this notebook, so the notebook is safe to share.

In [2]:
# @title Load the API key
if IN_COLAB:
    from google.colab import userdata
    try:
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        pass

from medsimp import config
if config.OPENROUTER_API_KEY:
    print(f"API key found. Main model: {config.OPENROUTER_MODEL}")
    print(f"Fallbacks if it's busy: {', '.join(config.OPENROUTER_FALLBACK_MODELS)}")
else:
    print("No API key found. Parts 1 and 2 still work; add the key (see above) for Part 3.")

# Optional: keep LLM replies in Google Drive so re-running tomorrow doesn't use up free requests.
USE_DRIVE_CACHE = False  # @param {type:"boolean"}
if IN_COLAB and USE_DRIVE_CACHE:
    from pathlib import Path
    from google.colab import drive
    drive.mount("/content/drive")
    config.CACHE_DIR = Path("/content/drive/MyDrive/med-label-simplifier-cache")
    print("Caching replies in", config.CACHE_DIR)

API key found. Main model: google/gemma-4-31b-it:free
Fallbacks if it's busy: qwen/qwen3.8-27b:free, nvidia/nemotron-3-super-120b-a12b:free


## Part 1: The fact checker (no API key needed)

Here is the real *Directions* section from an ibuprofen label (from the openFDA API), and the critical facts the extractor finds in it. Note that openFDA strips punctuation from lists, so the original is one long run-on "sentence".

In [3]:
import pandas as pd
from medsimp.facts import compare_facts, extract_facts

original = (
    "Directions do not take more than directed the smallest effective dose should be used adults and children "
    "12 years and over: take 1 tablet every 4 to 6 hours while symptoms persist if pain or fever does not "
    "respond to 1 tablet, 2 tablets may be used do not exceed 6 tablets in 24 hours, unless directed by a "
    "doctor children under 12 years: ask a doctor"
)
pd.DataFrame([{"found in text": f.text, "normalised as": f.key, "type": f.kind} for f in extract_facts(original)])

,found in text,normalised as,type
0,12 years,12 year,quantity
1,1 tablet,1 tablet,quantity
2,4 to 6 hours,4-6 hour,quantity
3,2 tablets,2 tablet,quantity
4,6 tablets,6 tablet,quantity
5,24 hours,24 hour,quantity


Now two rewrites. The **good** one keeps every fact. The **bad** one reads nicely, but it drops the 24-hour limit and invents a daily maximum of 8 tablets. This is exactly the kind of mistake that could hurt someone.

In [4]:
good = (
    "If you are 12 years or older, take 1 tablet every 4 to 6 hours while you have symptoms. "
    "If 1 tablet does not help, you may take 2 tablets. "
    "Do not take more than 6 tablets in 24 hours, unless a doctor tells you to. "
    "If you are younger than 12 years, ask a doctor."
)
bad = "Take 1 tablet every 4 to 6 hours. If it doesn't help, take 2 tablets. You can take up to 8 tablets a day."

for name, rewrite in [("GOOD rewrite", good), ("BAD rewrite", bad)]:
    check = compare_facts(original, rewrite)
    print(f"{name}: kept {len(check.kept)}/{len(check.kept) + len(check.missing)} facts ({check.recall:.0%} recall)")
    for f in check.missing:
        print(f"   MISSING:  {f.text}")
    for f in check.invented:
        print(f"   INVENTED: {f.text}")
    print()

GOOD rewrite: kept 6/6 facts (100% recall)

BAD rewrite: kept 3/6 facts (50% recall)
   MISSING:  12 years
   MISSING:  6 tablets
   MISSING:  24 hours
   INVENTED: 8 tablets



## Part 2: The meaning check catches a real hallucination (no API key needed)

While building this project, the LLM rewrote the ibuprofen pregnancy warning and **added "It may cause liver problems"**. The original never says that.

The fact checker **missed** it, because the word "liver" does appear elsewhere in the label ("liver cirrhosis"). The NLI meaning check **caught** it. That's why the project uses two independent checks.

The NLI model (`cross-encoder/nli-deberta-v3-small`) runs locally. The first run downloads it (about 500 MB).

In [5]:
from medsimp.verify import check_meaning

original_warning = (
    "It is especially important not to use ibuprofen at 20 weeks or later in pregnancy unless definitely "
    "directed to do so by a doctor because it may cause problems in the unborn child or complications during "
    "delivery. Keep out of reach of children. In case of overdose, get medical help or contact a Poison "
    "Control Center right away."
)
llm_rewrite = (  # what the LLM actually produced (shortened)
    "Do not use ibuprofen at 20 weeks or later if you are pregnant unless a doctor tells you to. "
    "It may cause problems for the unborn child. It may cause liver problems. "
    "Keep out of reach of children. "
    "If you take too much medicine, get medical help or call a Poison Control Center right away."
)

facts = compare_facts(original_warning, llm_rewrite)
print(f"Fact check: {len(facts.kept)}/{len(facts.kept) + len(facts.missing)} facts kept, "
      f"{len(facts.invented)} invented numbers  ->  looks fine...")
result = check_meaning(original_warning, llm_rewrite)
print(f"\nNLI faithfulness: {result.faithfulness:.0%} of simplified sentences are supported by the original")
for flag in result.flags:
    if flag.direction == "unsupported":
        print(f"   NOT SUPPORTED by the original: {flag.sentence}")

Fact check: 5/5 facts kept, 0 invented numbers  ->  looks fine...


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]


NLI faithfulness: 80% of simplified sentences are supported by the original
   NOT SUPPORTED by the original: It may cause liver problems.


In [6]:
# And a dangerous contradiction:
wrong = "It is safe to use ibuprofen at 20 weeks or later in pregnancy."
for flag in check_meaning(original_warning, wrong).hard_contradictions:
    print(f"CONTRADICTION ({flag.score:.0%} confident): {flag.sentence}")

CONTRADICTION (100% confident): It is safe to use ibuprofen at 20 weeks or later in pregnancy.


## Part 3: The full pipeline on a real label (needs API key)

Fetch the ibuprofen label from openFDA → simplify each section with the LLM → fact check + meaning check → retry if needed → report.

**Legend:** green = critical fact kept, red = critical fact missing, orange = number that wasn't in the original.

Free OpenRouter models allow 50 requests per day, and this uses about 3–6.

In [7]:
from medsimp.pipeline import simplify_drug
from medsimp.render import drug_html

CORE = ["dosage_and_administration", "warnings", "stop_use"]

if not config.OPENROUTER_API_KEY:
    print("Add your OpenRouter key in the Setup section to run this part.")
else:
    ibuprofen = simplify_drug("ibuprofen", sections=CORE, on_progress=print)
    display(HTML(drug_html(ibuprofen)))

Simplifying Dosage and administration...


Simplifying Warnings...


Simplifying Stop use...


## Part 4: Try any medicine (needs API key)

Type a generic or brand name (e.g. `loratadine`, `Tylenol`, `omeprazole`, `metformin`) and run the cell.

In [8]:
DRUG = "loratadine"  # @param {type:"string"}
ALL_SECTIONS = False  # @param {type:"boolean"}
USE_NLI = True  # @param {type:"boolean"}

if not config.OPENROUTER_API_KEY:
    print("Add your OpenRouter key in the Setup section to run this part.")
else:
    sections = None if ALL_SECTIONS else CORE + ["boxed_warning", "contraindications"]
    result = simplify_drug(DRUG, use_nli=USE_NLI, sections=sections, on_progress=print)
    display(HTML(drug_html(result)))

Simplifying Dosage and administration...


Simplifying Warnings...


Simplifying Stop use...


## Part 5: Evaluation across many drugs (optional, needs API key)

This runs the pipeline on 12 common over-the-counter drugs and measures:
- **critical-fact recall** on the first try vs. after the verify-and-retry loop
- invented numbers
- readability before and after

It needs roughly 70–100 LLM requests, which is more than one day of the free tier. Turn on `USE_DRIVE_CACHE` in Setup and run it on two days; it continues where it stopped.

In [9]:
RUN_EVALUATION = False  # @param {type:"boolean"}

if RUN_EVALUATION:
    !python evaluate.py
    df = pd.read_csv("results/eval.csv")
    total = df.facts_total.sum()
    print(f"\n{df.drug.nunique()} drugs, {len(df)} sections")
    print(f"Fact recall, first try:          {(df.first_try_recall * df.facts_total).sum() / total:.1%}")
    print(f"Fact recall, after verify+retry: {df.facts_kept.sum() / total:.1%}")
    print(f"Invented numbers, first try → final: {df.first_try_invented.sum()} → {df.invented_facts.sum()}")
    print(f"Hard words: {df.hard_words_before.mean():.1f}% → {df.hard_words_after.mean():.1f}%")
    print("\nFor charts and error analysis, open notebooks/evaluation.ipynb")
else:
    print("Tick RUN_EVALUATION and run this cell to start the evaluation.")

Tick RUN_EVALUATION and run this cell to start the evaluation.


## How it works

```
openFDA label ──► split into sections ──► LLM simplifies (grade 5)
                                                │
                  ┌─────────────────────────────┘
                  ▼
           Fact check: every dose / time / age / warning kept? any invented numbers?
           Meaning check (NLI): each sentence supported by the other text? contradictions?
                  │
      problems? ──┴── yes ──► "fix-it" prompt listing the exact problems ──► retry (max 2)
                  │
                  no ──► report readability + metrics
```

Key design choices (details in the [README](https://github.com/Pranavladha612/med-label-simplifier#design-notes-and-early-findings)):
- **Retrieve, then verify.** Small NLI models fail on paragraphs, so each sentence is compared with the 3 most similar sentences on the other side.
- **Only reliable signals trigger retries.** An overly strict checker made the LLM copy jargon back in, so softer NLI flags are shown for human review instead.
- **Fair readability.** openFDA strips list punctuation, which inflates reading-grade scores, so "% hard words" is reported alongside them.